# Etapa II · Modelos finales: selección paso a paso y guardado

Este notebook recorre las decisiones de los dos modelos:

1. **Supervisado:** clasificador, normalización, representación, n-gramas, SVD e hiperparámetros.
2. **No supervisado:** `k`, normalización, representación (incluida la binaria), n-gramas, SVD e hiperparámetros.

Al final guarda en `models/etapa2/` los dos modelos ya ajustados con train+dev, listos para
usarse con texto nuevo. **No vuelve a evaluar test**: lee las métricas congeladas del
informe.

In [1]:
%matplotlib inline

In [2]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
import sys
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer
from IPython.display import display

from src.etapa2.classify import make_vectorizer
from src.etapa2.labels import load_labeled
from src.etapa2.normalization import load_variants
from src.normalize import get_nlp, normalize_doc

pd.set_option("display.width", 180)
print("Raíz:", ROOT)


Raíz: /home/mrangello/Documentos/PracticaML


In [3]:
exp = pd.read_csv(ROOT / "results/etapa2/experiments.tsv", sep="\t",
                   dtype=str, keep_default_na=False)

def get(experiment_id):
    return exp[exp["experiment_id"] == experiment_id].iloc[0]

def dev_macro_f1(row):
    return float(row["dev_metric"].split("macro_f1=")[1].split(";")[0])

def silhouette(row):
    return float(row["dev_metric"].split("=")[1])

def extra(row):
    return dict(part.split("=", 1) for part in row["extra"].split("; "))


## Parte 1 · Modelo supervisado

Cada paso cambia un factor y arrastra la mejor opción del paso anterior. La métrica de
decisión es el macro-F1 en dev; si hay empate (menos de un punto), se desempata por CV y
simplicidad.

In [4]:
steps = [
    ("Clasificador", "C-1", ["C01", "C02", "C03"], "C02",
     "LR gana a NB y SVC"),
    ("Normalización", "C-2", ["C05", "C08", "C02"], "C02",
     "N2; N1 empata en dev pero tiene CV menor"),
    ("Representación", "C-3", ["C12", "C13", "C02"], "C02",
     "TF-IDF gana a binaria y frecuencia"),
    ("N-gramas", "C-4", ["C02", "C18", "C19"], "C02",
     "(1,1); C18 gana 0.0014 pero con CV menor"),
    ("SVD", "C-5", ["C02", "C28", "C26", "C27"], "C02",
     "sin SVD; todas las variantes bajan"),
]
rows = []
for step, block, ids, chosen, why in steps:
    candidates = ", ".join(f"{i}={dev_macro_f1(get(i)):.4f}" for i in ids)
    rows.append({"Paso": step, "Bloque": block, "Candidatos (dev)": candidates,
                 "Elegido": chosen, "Motivo": why})
display(pd.DataFrame(rows))

,Paso,Bloque,Candidatos (dev),Elegido,Motivo
0,Clasificador,C-1,"C01=0.7607, C02=0.7993, C03=0.7481",C02,LR gana a NB y SVC
1,Normalización,C-2,"C05=0.7602, C08=0.7981, C02=0.7993",C02,N2; N1 empata en dev pero tiene CV menor
2,Representación,C-3,"C12=0.7445, C13=0.7457, C02=0.7993",C02,TF-IDF gana a binaria y frecuencia
3,N-gramas,C-4,"C02=0.7993, C18=0.8007, C19=0.7751",C02,"(1,1); C18 gana 0.0014 pero con CV menor"
4,SVD,C-5,"C02=0.7993, C28=0.7706, C26=0.7586, C27=0.7475",C02,sin SVD; todas las variantes bajan


El clasificador queda:

```text
N2 → TF-IDF (1,1) → LogisticRegression(C=1, min_df=1)
```

Ese es **C02**: dev 0.7993 y test 0.7610. Ahora lo reajusto con train+dev y lo guardo.

In [5]:
df = load_labeled()
texts = load_variants().set_index("document_id")["text_n2"]
train_dev = df[df["split"].isin(["train", "dev"])]
X_train_dev = texts.loc[train_dev["document_id"]].tolist()
y_train_dev = train_dev["label"].tolist()

clf = Pipeline([
    ("vec", make_vectorizer("tfidf", (1, 1))),
    ("clf", LogisticRegression(max_iter=2000, random_state=0)),
]).set_params(vec__min_df=1, clf__C=1.0)
clf.fit(X_train_dev, y_train_dev)

model_dir = ROOT / "models" / "etapa2"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": clf, "config": {
    "experiment": "C02", "normalization": "N2", "representation": "tfidf",
    "ngram_range": [1, 1], "C": 1.0, "min_df": 1,
}}, model_dir / "classifier_c02.pkl")
print("Guardado", model_dir / "classifier_c02.pkl")

Guardado /home/mrangello/Documentos/PracticaML/models/etapa2/classifier_c02.pkl


## Parte 2 · Modelo no supervisado

Aquí no hay etiquetas. La decisión se apoya en silhouette en dev, estabilidad e
interpretabilidad. La binaria se añadió como **K17** para completar las tres
representaciones.

In [6]:
steps = [
    ("k", "K-1", ["K01", "K02", "K03", "K04", "K05"], "K02",
     "k=4: silhouette empatado y 4 clusters temáticos"),
    ("Normalización", "K-2", ["K06", "K07", "K02"], "K02",
     "N2: N0 sube silhouette pero ARI ~0.02"),
    ("Representación", "K-3", ["K17", "K08", "K02"], "K02",
     "TF-IDF: binaria 0.0477 y frecuencia separa formato"),
    ("N-gramas", "K-4", ["K02", "K09"], "K02",
     "unigramas: bigramas separa números/LaTeX"),
    ("SVD", "K-5", ["K10", "K11", "K02", "K12"], "K02",
     "SVD 100: 50/200 inflan silhouette con formato"),
    ("Hiperparámetros", "K-6", ["K02", "K13", "K14", "K15", "K16"], "K02",
     "init/n_init/max_iter no mejoran"),
]
rows = []
for step, block, ids, chosen, why in steps:
    candidates = ", ".join(
        f"{i}={silhouette(get(i)):.4f}/ARI {float(extra(get(i))['ari_dev']):.3f}"
        for i in ids)
    rows.append({"Paso": step, "Bloque": block, "Candidatos (silhouette/ARI dev)": candidates,
                 "Elegido": chosen, "Motivo": why})
display(pd.DataFrame(rows))

,Paso,Bloque,Candidatos (silhouette/ARI dev),Elegido,Motivo
0,k,K-1,"K01=0.0651/ARI 0.285, K02=0.0660/ARI 0.376, K0...",K02,k=4: silhouette empatado y 4 clusters temáticos
1,Normalización,K-2,"K06=0.0986/ARI 0.022, K07=0.0546/ARI 0.330, K0...",K02,N2: N0 sube silhouette pero ARI ~0.02
2,Representación,K-3,"K17=0.0477/ARI -0.004, K08=0.1727/ARI 0.043, K...",K02,TF-IDF: binaria 0.0477 y frecuencia separa for...
3,N-gramas,K-4,"K02=0.0660/ARI 0.376, K09=0.1873/ARI 0.035",K02,unigramas: bigramas separa números/LaTeX
4,SVD,K-5,"K10=0.0584/ARI 0.060, K11=0.2085/ARI 0.043, K0...",K02,SVD 100: 50/200 inflan silhouette con formato
5,Hiperparámetros,K-6,"K02=0.0660/ARI 0.376, K13=0.2085/ARI 0.043, K1...",K02,init/n_init/max_iter no mejoran


La configuración final es:

```text
N2 → TF-IDF (1,1) → SVD 100 → Normalizer → KMeans(k=4, k-means++, n_init=10)
```

Ese es **K02**: silhouette test 0.0829, ARI 0.3719, NMI 0.4860. Ahora lo reajusto con
train+dev y lo guardo.

In [7]:
km = Pipeline([
    ("vec", make_vectorizer("tfidf", (1, 1))),
    ("svd", TruncatedSVD(n_components=100, random_state=0)),
    ("l2", Normalizer()),
    ("km", KMeans(n_clusters=4, init="k-means++", n_init=10,
                  max_iter=300, random_state=0)),
])
km.fit(X_train_dev)
joblib.dump({"pipeline": km, "config": {
    "experiment": "K02", "normalization": "N2", "representation": "tfidf",
    "ngram_range": [1, 1], "svd_components": 100, "k": 4,
    "init": "k-means++", "n_init": 10, "max_iter": 300,
}}, model_dir / "kmeans_k02.pkl")
print("Guardado", model_dir / "kmeans_k02.pkl")

Guardado /home/mrangello/Documentos/PracticaML/models/etapa2/kmeans_k02.pkl


## Parte 3 · Usar los modelos guardados

Los pipelines guardados reciben **texto ya normalizado** (N2), no `document_id`. Para un
texto nuevo, primero se normaliza con `normalize_doc`.

In [8]:
import sklearn

metadata = {
    "classifier": {
        "experiment": "C02", "normalization": "N2", "representation": "tfidf",
        "ngram_range": [1, 1], "C": 1.0, "min_df": 1,
        "dev_macro_f1": 0.7993, "test_macro_f1": 0.7610,
    },
    "kmeans": {
        "experiment": "K02", "normalization": "N2", "representation": "tfidf",
        "ngram_range": [1, 1], "svd_components": 100, "k": 4,
        "init": "k-means++", "n_init": 10, "max_iter": 300,
        "test_silhouette": 0.0829, "test_ari": 0.3719, "test_nmi": 0.4860,
    },
    "versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__, "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "random_state": 0,
}
with open(model_dir / "metadata.json", "w", encoding="utf-8") as fh:
    json.dump(metadata, fh, ensure_ascii=False, indent=2)
print("Guardado", model_dir / "metadata.json")

Guardado /home/mrangello/Documentos/PracticaML/models/etapa2/metadata.json


In [9]:
clf_loaded = joblib.load(model_dir / "classifier_c02.pkl")["pipeline"]
km_loaded = joblib.load(model_dir / "kmeans_k02.pkl")["pipeline"]

sample = ("Retrieval-augmented generation for multi-hop question answering with "
          "large language models. We propose a retrieval pipeline that selects "
          "evidence documents and improves answer accuracy.")
norm = normalize_doc(get_nlp()(sample))
print("Normalizado:", norm[:120], "...")

pred = clf_loaded.predict([norm])[0]
proba = clf_loaded.predict_proba([norm])[0]
print("\nClase predicha:", pred)
print("Probabilidades:", sorted(zip(clf_loaded.classes_, proba), key=lambda x: -x[1]))
print("Cluster:", int(km_loaded.predict([norm])[0]))

Normalizado: retrieval augment generation multi hop question answer large language model propose retrieval pipeline select evidence d ...

Clase predicha: cs.IR
Probabilidades: [(np.str_('cs.IR'), np.float64(0.5180435559366164)), (np.str_('cs.CL'), np.float64(0.2761211718942012)), (np.str_('cs.CV'), np.float64(0.12015884061333124)), (np.str_('cs.LG'), np.float64(0.0856764315558511))]
Cluster: 1


In [10]:
for path in sorted(model_dir.glob("*")):
    print(f"{path.name:22} {path.stat().st_size / 1024:8.1f} KB")

classifier_c02.pkl        389.6 KB
kmeans_k02.pkl           5969.6 KB
metadata.json               0.7 KB


## Resumen

- **Supervisado:** C02 = N2 + TF-IDF (1,1) + LR(C=1, min_df=1). Test macro-F1 0.761.
- **No supervisado:** K02 = N2 + TF-IDF (1,1) + SVD 100 + KMeans(k=4). Test silhouette 0.083,
  ARI 0.372, NMI 0.486.
- **K17 binaria:** silhouette dev 0.0477 y ARI −0.0038; no mejora a TF-IDF.
- Los modelos quedan en `models/etapa2/` y se cargan con `joblib.load`.
- Test no se volvió a evaluar: las métricas son las congeladas del informe.